# Model A: Custom CNN (v3)

Binary classification of **Real vs AI-Generated** images with a CNN trained from scratch.
Dataset: Defactify Image Dataset (group-based split from `01_eda_data_preparation_fix`).
Framework: TensorFlow / Keras. Runs on Kaggle or Colab (auto-detected).

## What changed compared with v2
| Area | v2 | v3 |
|---|---|---|
| Environment | Kaggle only (`kaggle_secrets` imported at the top) | Kaggle or Colab, auto-detected |
| Resize | squash to 224x224 (real photos are 4:3, AI images are square, so aspect ratio leaked the label) | centered square crop, then resize |
| Augmentation | none | training-only: flip, random crop, JPEG recompression (both classes) |
| Epochs | 10 (validation loss still falling) | up to 20, EarlyStopping patience 5 |
| Outputs | metrics only | + ROC curve, accuracy per generator, saved validation probabilities |
| Filenames | `custom_cnn_*` | `custom_cnn_v3_*` (v2 files are kept as a baseline for the scenario comparison) |

The test split is **not** used in this notebook. The decision threshold is chosen on validation and frozen for `final_testing_comparison`.

## 1. Environment

In [ ]:
!pip -q install datasets huggingface_hub

In [ ]:
import os
import io
import json
import shutil
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

# Compatibility shim for loading nested Keras models on some Colab/Kaggle images
if not hasattr(keras, "KerasTensor"):
    from keras.src.backend import KerasTensor
    keras.KerasTensor = KerasTensor

from pathlib import Path
from PIL import Image as PILImage
from datasets import load_dataset
from tensorflow.keras import layers, models

In [ ]:
# ----------------------------------------------------------------------------
# EXPERIMENT CONFIGURATION  (change only this cell between runs)
# ----------------------------------------------------------------------------
RUN_TAG = "v3"                 # part of every output filename; change it for ablation runs
MODEL_NAME = "custom_cnn"  # filename prefix

# Shared data-pipeline switches (identical in Model A and Model B notebooks)
CENTER_CROP = True   # crop to a centered square before resizing (no aspect-ratio distortion)
USE_AUG = True       # training-only augmentation (flip, random crop, JPEG recompression)
JPEG_PROB = 0.5      # probability of JPEG recompression, applied to BOTH classes
JPEG_QUALITY = (60, 95)
CROP_FRACTION = (0.80, 1.00)   # random crop side, as a fraction of the square side

IMAGE_SIZE = 224
IMAGE_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, 3)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

# Model A training
EPOCHS = 20
ES_PATIENCE = 5
RLR_PATIENCE = 2
RUN_TUNING = True
TUNE_TRAIN_SIZE = 6000
TUNE_VAL_SIZE = 1500
TUNE_EPOCHS = 2
DEFAULT_CONFIG = {"learning_rate": 3e-4, "dropout": 0.3}
tuning_grid = [
    {"learning_rate": 1e-3, "dropout": 0.3},
    {"learning_rate": 1e-3, "dropout": 0.5},
    {"learning_rate": 3e-4, "dropout": 0.3},
    {"learning_rate": 3e-4, "dropout": 0.5},
]

DATASET_ID = "Rajarshi-Roy-research/Defactify_Image_Dataset"
# Pinned dataset commit (the manifest in the repo still says "main")
HF_REVISION_FALLBACK = "1c3b714c9691eb6dd11c2c1e92eab1ac29584fa0"

REPO_URL = "https://github.com/Michaelatj/AI-Generated-Image-Classification-using-CNN.git"
GIT_USER_NAME = "Michael Andreas Tjendra"
GIT_USER_EMAIL = "alongbest2000@gmail.com"
GITHUB_USERNAME = "Michaelatj"

# Environment detection (Kaggle or Colab)
IN_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IN_KAGGLE else Path("/content")
HF_CACHE_DIR = "/kaggle/temp/huggingface_cache" if IN_KAGGLE else "/content/huggingface_cache"
REPO_DIR = WORK_DIR / "AI-Generated-Image-Classification-using-CNN"

print("Environment:", "Kaggle" if IN_KAGGLE else "Colab/other")
print("Run tag:", RUN_TAG, "| CENTER_CROP:", CENTER_CROP, "| USE_AUG:", USE_AUG)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

status = subprocess.check_output(["git", "status", "--porcelain"], text=True)
if status.strip():
    raise RuntimeError("Repository contains local changes. Check git status before syncing.")

subprocess.run(["git", "pull", "--rebase", "origin", "main"], check=True)
print(subprocess.check_output(["git", "log", "--oneline", "-n", "5"], text=True))

ARTIFACT_DIR = REPO_DIR / "artifacts"
SPLIT_DIR = ARTIFACT_DIR / "splits"
CONFIG_DIR = ARTIFACT_DIR / "configs"
MODEL_DIR = ARTIFACT_DIR / "models"
EVAL_DIR = ARTIFACT_DIR / "evaluation"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("Splits:", sorted(p.name for p in SPLIT_DIR.glob("*")))

In [ ]:
tf.keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as error:
        print(error)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Available GPU:", gpus)
if not gpus:
    print("WARNING: no GPU found. Enable a GPU accelerator before training.")

total, used, free = shutil.disk_usage(str(WORK_DIR))
print(f"Free storage: {free / (1024**3):.2f} GB")

## 2. Dataset and split verification
Loads the shared manifests, then verifies before any training: no caption-group overlap, no row used twice, and manifest labels/captions equal the original Hugging Face rows.

In [ ]:
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv")

# Same ordering rule for every split, so prediction order always matches the dataframe
sort_cols = ["source_split", "row_idx"]
train_df = train_df.sort_values(sort_cols).reset_index(drop=True)
val_df = val_df.sort_values(sort_cols).reset_index(drop=True)
test_df = test_df.sort_values(sort_cols).reset_index(drop=True)

SPLITS = {"train": train_df, "validation": val_df, "test": test_df}

for name, df in SPLITS.items():
    print(f"{name}: {df.shape} | class share:",
          df["label_a"].value_counts(normalize=True).round(3).to_dict())

# ---- leakage checks -------------------------------------------------------
groups = {n: set(df["caption_group_id"].dropna()) for n, df in SPLITS.items()}
keys = {n: set(zip(df["source_split"], df["row_idx"])) for n, df in SPLITS.items()}

for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    assert not (groups[a] & groups[b]), f"LEAKAGE: {a} and {b} share caption groups"
    assert not (keys[a] & keys[b]), f"Same source row in {a} and {b}"
    print(f"{a} vs {b}: caption-group overlap = 0, row overlap = 0")

for name, df in SPLITS.items():
    assert df["image_id"].is_unique, f"duplicate image_id in {name}"
    assert not df[sort_cols].duplicated().any(), f"duplicate source rows in {name}"
    assert set(df["label_a"].unique()).issubset({0, 1})
print("Split checks passed.")

In [ ]:
manifest_path = CONFIG_DIR / "dataset_manifest.json"
HF_REVISION = None
if manifest_path.exists():
    HF_REVISION = json.load(open(manifest_path, encoding="utf-8")).get("dataset_revision")
if not HF_REVISION or HF_REVISION == "main":
    HF_REVISION = HF_REVISION_FALLBACK
print("Dataset revision:", HF_REVISION)

HF_DATASETS = {}
# Only splits that training/validation need. The test split is NOT touched here.
required_splits = sorted(set(train_df["source_split"]) | set(val_df["source_split"]))
for split_name in required_splits:
    HF_DATASETS[split_name] = load_dataset(
        DATASET_ID, split=split_name, revision=HF_REVISION, cache_dir=HF_CACHE_DIR
    )
    print(f"{split_name}: {len(HF_DATASETS[split_name]):,} images")


def normalize_caption(text):
    return " ".join(str(text).strip().split()).casefold()


def validate_mapping(df, split_name):
    for source_split, group in df.groupby("source_split", sort=False):
        hf = HF_DATASETS[source_split]
        idx = group["row_idx"].astype(int).tolist()
        assert min(idx) >= 0 and max(idx) < len(hf), f"Invalid row_idx in {split_name}"
        selected = hf.select(idx)   # does not decode the images
        assert np.array_equal(np.asarray(selected["Label_A"]), group["label_a"].to_numpy()), f"Label_A mismatch: {split_name}"
        assert np.array_equal(np.asarray(selected["Label_B"]), group["label_b"].to_numpy()), f"Label_B mismatch: {split_name}"
        assert [normalize_caption(x) for x in selected["Caption"]] == [normalize_caption(x) for x in group["caption"]], f"Caption mismatch: {split_name}"
    print(f"{split_name}: manifest rows match Hugging Face rows")


validate_mapping(train_df, "train")
validate_mapping(val_df, "validation")

## 3. Preprocessing and data pipeline
The same functions are used in Model B, in `final_testing_comparison`, and in the deployed app. Images are fetched by `source_split` + `row_idx`, so no copy of the dataset is stored.

In [ ]:
# ----------------------------------------------------------------------------
# Shared preprocessing.  The deployed app MUST apply exactly prepare_image(..., training=False).
# ----------------------------------------------------------------------------
class_counts = train_df["label_a"].value_counts().reindex([0, 1], fill_value=0)
if (class_counts == 0).any():
    raise ValueError("A training class is missing!")
class_weights = {int(l): float(len(train_df) / (2 * class_counts[l])) for l in [0, 1]}
print("Class counts:", class_counts.to_dict())
print("Class weights:", class_weights)


def center_square(image):
    w, h = image.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    return image.crop((left, top, left + s, top + s))


def augment(image, rng):
    """Training-only. Applied identically to Real and AI-Generated images."""
    if rng.random() < 0.5:
        image = image.transpose(PILImage.Transpose.FLIP_LEFT_RIGHT)

    w, h = image.size
    frac = rng.uniform(*CROP_FRACTION)
    cw, ch = max(1, int(w * frac)), max(1, int(h * frac))
    left = int(rng.integers(0, w - cw + 1))
    top = int(rng.integers(0, h - ch + 1))
    image = image.crop((left, top, left + cw, top + ch))

    # JPEG recompression: removes the "PNG vs JPEG" and compression-history shortcut
    if rng.random() < JPEG_PROB:
        quality = int(rng.integers(JPEG_QUALITY[0], JPEG_QUALITY[1] + 1))
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)
        image = PILImage.open(buffer).convert("RGB")
    return image


def prepare_image(image, training=False, rng=None):
    image = image.convert("RGB")
    if CENTER_CROP:
        image = center_square(image)
    if training and USE_AUG:
        image = augment(image, rng)
    return image.resize((IMAGE_SIZE, IMAGE_SIZE), resample=PILImage.Resampling.BILINEAR)


def make_generator(df, training, seed):
    rng = np.random.default_rng(seed)

    def gen():
        # full permutation of the training rows on every pass
        d = df.iloc[rng.permutation(len(df))].reset_index(drop=True) if training else df
        for row in d.itertuples(index=False):
            image = HF_DATASETS[row.source_split][int(row.row_idx)]["Image"]
            if image is None:
                raise ValueError(f"Missing image: {row.image_id}")
            arr = np.asarray(prepare_image(image, training, rng), dtype=np.uint8)
            label = np.array([int(row.label_a)], dtype=np.float32)
            weight = np.float32(class_weights[int(row.label_a)] if training else 1.0)
            yield arr, label, weight

    return gen


OUTPUT_SIGNATURE = (
    tf.TensorSpec(shape=IMAGE_SHAPE, dtype=tf.uint8),
    tf.TensorSpec(shape=(1,), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.float32),
)


def create_tf_dataset(df, training=False, seed=SEED, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_generator(
        make_generator(df, training, seed), output_signature=OUTPUT_SIGNATURE
    )
    ds = ds.apply(tf.data.experimental.assert_cardinality(len(df)))
    return ds.batch(batch_size).prefetch(AUTOTUNE)


train_dataset = create_tf_dataset(train_df, training=True)
val_dataset = create_tf_dataset(val_df, training=False)

images, labels, weights = next(iter(train_dataset))
print("Batch:", images.shape, images.dtype, "| labels:", labels.shape, "| weights:", weights.shape)

In [ ]:
# Visual check of the pipeline (use these figures in the report: preprocessing + augmentation)
rng_vis = np.random.default_rng(0)
rows = train_df.sample(n=4, random_state=SEED)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for j, (_, row) in enumerate(rows.iterrows()):
    img = HF_DATASETS[row["source_split"]][int(row["row_idx"])]["Image"].convert("RGB")
    name = "Real" if row["label_a"] == 0 else "AI-Generated"
    axes[0, j].imshow(img)
    axes[0, j].set_title(f"Original: {name}\n{img.size[0]}x{img.size[1]}")
    axes[1, j].imshow(prepare_image(img, training=False))
    axes[1, j].set_title("Eval pipeline (224x224)")
    axes[2, j].imshow(prepare_image(img, training=True, rng=rng_vis))
    axes[2, j].set_title("Training pipeline (augmented)")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.savefig(EVAL_DIR / f"{MODEL_NAME}_{RUN_TAG}_pipeline_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Custom CNN architecture
Input 224x224x3 (raw pixels) -> Rescaling -> 3 blocks (32/64/128 filters, 2x Conv-BN-ReLU + MaxPool) -> GlobalAveragePooling -> Dense(128) -> Dropout -> Dense(1, sigmoid = P(AI-Generated)).

In [ ]:
def conv_block(x, filters, dropout_rate=0.0):
    for _ in range(2):
        x = layers.Conv2D(filters, kernel_size=3, padding="same",
                          use_bias=False, kernel_initializer="he_normal")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    return x


def build_model(head_dropout=0.4):
    inputs = layers.Input(shape=IMAGE_SHAPE, name="image")
    x = layers.Rescaling(1.0 / 255, name="rescale")(inputs)   # raw 0-255 RGB in, 0-1 inside
    x = conv_block(x, 32)
    x = conv_block(x, 64, dropout_rate=0.10)
    x = conv_block(x, 128, dropout_rate=0.20)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(head_dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="prediction")(x)
    return models.Model(inputs, outputs, name="Custom_CNN")


model = build_model()
model.summary()

In [ ]:
def compile_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )
    return model

## Hyperparameter tuning (small subset)
Short runs on random subsets of train/validation to choose the learning rate and head dropout. Same procedure as Model B.

In [ ]:
tuning_df = pd.DataFrame()
best_config = dict(DEFAULT_CONFIG)

if RUN_TUNING:
    # Small random subsets taken from their own splits (no leakage)
    tune_train = (train_df.sample(n=min(TUNE_TRAIN_SIZE, len(train_df)), random_state=SEED)
                  .sort_values(sort_cols).reset_index(drop=True))
    tune_val = (val_df.sample(n=min(TUNE_VAL_SIZE, len(val_df)), random_state=SEED)
                .sort_values(sort_cols).reset_index(drop=True))
    tune_train_ds = create_tf_dataset(tune_train, training=True)
    tune_val_ds = create_tf_dataset(tune_val, training=False)

    rows = []
    for cfg in tuning_grid:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)
        m = compile_model(build_model(cfg["dropout"]), cfg["learning_rate"])
        h = m.fit(tune_train_ds, validation_data=tune_val_ds, epochs=TUNE_EPOCHS, verbose=0)
        i = int(np.argmin(h.history["val_loss"]))
        rows.append({**cfg,
                     "best_val_loss": h.history["val_loss"][i],
                     "val_accuracy": h.history["val_accuracy"][i],
                     "val_auc": h.history["val_auc"][i]})
        print(rows[-1])

    tuning_df = pd.DataFrame(rows).sort_values("best_val_loss").reset_index(drop=True)
    display(tuning_df)
    best_config = {"learning_rate": float(tuning_df.loc[0, "learning_rate"]),
                   "dropout": float(tuning_df.loc[0, "dropout"])}

print("Selected config:", best_config)

## Training
Class weights handle the 5:1 imbalance. The training accuracy/loss in Keras are therefore **sample-weighted**, while validation metrics are unweighted. Compare the shape of the curves, not the absolute values.

In [ ]:
BEST_MODEL_PATH = MODEL_DIR / f"{MODEL_NAME}_{RUN_TAG}_best.keras"
HISTORY_PATH = MODEL_DIR / f"{MODEL_NAME}_{RUN_TAG}_history.csv"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(BEST_MODEL_PATH), monitor="val_loss", mode="min",
                                       save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=ES_PATIENCE,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=RLR_PATIENCE,
                                         min_lr=1e-6, verbose=1),
]

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

model = build_model(best_config["dropout"])
compile_model(model, best_config["learning_rate"])

history = model.fit(train_dataset, validation_data=val_dataset,
                    epochs=EPOCHS, callbacks=callbacks, verbose=1)
print("Training completed. Epochs run:", len(history.epoch))

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.index.name = "epoch"
history_df.to_csv(HISTORY_PATH)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (tr, va, title) in zip(axes, [("accuracy", "val_accuracy", "Accuracy"), ("loss", "val_loss", "Loss")]):
    ax.plot(history_df[tr].values, label=f"Training {title} (sample-weighted)")
    ax.plot(history_df[va].values, label=f"Validation {title}")
    ax.set_title(f"Training vs Validation {title}")
    ax.set_xlabel("Epoch"); ax.set_ylabel(title); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
CURVES_PATH = EVAL_DIR / f"{MODEL_NAME}_{RUN_TAG}_training_curves.png"
plt.savefig(CURVES_PATH, dpi=150, bbox_inches="tight")
plt.show()
display(history_df)

## Validation evaluation
Threshold selection (max macro-F1) happens here, on validation only. The numbers at the tuned threshold are slightly optimistic; the unbiased numbers come from the test split in `final_testing_comparison`.

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             classification_report, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)


def evaluate_validation(model, prefix):
    """Validation-only evaluation. The threshold is chosen here and then frozen for the test split."""
    keras_metrics = model.evaluate(val_dataset, return_dict=True, verbose=0)

    y_true = val_df["label_a"].to_numpy().astype(int)
    y_prob = model.predict(val_dataset.map(lambda i, l, w: i), verbose=1).ravel()
    assert len(y_prob) == len(y_true)

    thresholds = np.arange(0.05, 0.96, 0.01)
    f1s = [f1_score(y_true, (y_prob >= t).astype(int), average="macro", zero_division=0)
           for t in thresholds]
    best_threshold = float(thresholds[int(np.argmax(f1s))])

    def summarize(t):
        p = (y_prob >= t).astype(int)
        return {
            "threshold": round(float(t), 2),
            "accuracy": float(accuracy_score(y_true, p)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, p)),
            "precision_ai": float(precision_score(y_true, p, zero_division=0)),
            "recall_ai": float(recall_score(y_true, p, zero_division=0)),
            "recall_real": float(recall_score(y_true, p, pos_label=0, zero_division=0)),
            "f1_ai": float(f1_score(y_true, p, zero_division=0)),
            "f1_macro": float(f1_score(y_true, p, average="macro", zero_division=0)),
            "confusion_matrix": confusion_matrix(y_true, p, labels=[0, 1]).tolist(),
        }

    metrics = {
        "keras_evaluate": {k: float(v) for k, v in keras_metrics.items()},
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "at_threshold_0.50": summarize(0.5),
        "at_tuned_threshold": summarize(best_threshold),
        "tuned_threshold": round(best_threshold, 2),
        "threshold_selection": "max macro-F1 on validation set",
    }

    print("VALIDATION RESULTS | ROC-AUC:", round(metrics["roc_auc"], 4))
    for key in ["at_threshold_0.50", "at_tuned_threshold"]:
        r = metrics[key]
        print(f"\n{key} (threshold={r['threshold']})\n"
              f"  accuracy={r['accuracy']:.4f}  balanced_acc={r['balanced_accuracy']:.4f}  macro_f1={r['f1_macro']:.4f}\n"
              f"  recall_real={r['recall_real']:.4f}  recall_ai={r['recall_ai']:.4f}  precision_ai={r['precision_ai']:.4f}")

    # classification report at the tuned threshold
    y_pred = (y_prob >= best_threshold).astype(int)
    report_df = pd.DataFrame(classification_report(
        y_true, y_pred, target_names=["Real", "AI-Generated"],
        output_dict=True, zero_division=0)).transpose()
    display(report_df)

    # accuracy per generator (Label_B) -> which generator is hardest to detect
    tmp = val_df.assign(pred=y_pred)
    per_gen = pd.DataFrame([
        {"label_b_name": name, "images": len(g),
         "accuracy": float((g["pred"] == g["label_a"]).mean())}
        for name, g in tmp.groupby("label_b_name")
    ])
    display(per_gen)

    # confusion matrix
    cm = np.array(metrics["at_tuned_threshold"]["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Real", "AI-Generated"]); ax.set_yticklabels(["Real", "AI-Generated"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"{MODEL_NAME} validation (threshold={best_threshold:.2f})")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.tight_layout()
    plt.savefig(EVAL_DIR / f"{prefix}_confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ROC curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f"AUC = {metrics['roc_auc']:.4f}")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
    plt.title(f"{MODEL_NAME} validation ROC"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(EVAL_DIR / f"{prefix}_roc_curve.png", dpi=150, bbox_inches="tight")
    plt.show()

    # save artifacts (val probabilities let the final notebook reuse them without re-predicting)
    paths = {
        "metrics": EVAL_DIR / f"{prefix}_metrics.json",
        "report": EVAL_DIR / f"{prefix}_classification_report.csv",
        "per_generator": EVAL_DIR / f"{prefix}_per_generator.csv",
        "val_probs": EVAL_DIR / f"{prefix}_val_probs.csv",
    }
    with open(paths["metrics"], "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=4)
    report_df.to_csv(paths["report"])
    per_gen.to_csv(paths["per_generator"], index=False)
    pd.DataFrame({"image_id": val_df["image_id"], "label_a": y_true,
                  "label_b_name": val_df["label_b_name"], "prob_ai": y_prob}
                 ).to_csv(paths["val_probs"], index=False)
    return metrics, paths

In [ ]:
best_model = tf.keras.models.load_model(BEST_MODEL_PATH)
PREFIX = f"{MODEL_NAME}_{RUN_TAG}"
val_metrics, eval_paths = evaluate_validation(best_model, PREFIX)

## Save configuration

In [ ]:
CONFIG_PATH = MODEL_DIR / f"{MODEL_NAME}_{RUN_TAG}_config.json"
TUNING_PATH = MODEL_DIR / f"{MODEL_NAME}_{RUN_TAG}_tuning.csv"

training_config = {
    "experiment": f"Model A - Custom CNN ({RUN_TAG})",
    "environment": "Kaggle" if IN_KAGGLE else "Colab/other",
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "dataset": DATASET_ID,
    "dataset_revision": HF_REVISION,
    "target": "label_a",
    "class_mapping": {"0": "Real", "1": "AI-Generated"},
    "preprocessing": {
        "image_size": IMAGE_SIZE, "color_mode": "RGB",
        "center_square_crop": CENTER_CROP, "resize_interpolation": "bilinear",
        "model_input": "raw 0-255 RGB uint8/float, 224x224x3; Rescaling(1/255) is inside the model",
    },
    "augmentation_training_only": {
        "enabled": USE_AUG, "horizontal_flip": 0.5, "random_crop_fraction": list(CROP_FRACTION),
        "jpeg_probability": JPEG_PROB, "jpeg_quality": list(JPEG_QUALITY),
    },
    "shuffle": "full permutation of training rows every epoch",
    "architecture": ("3 conv blocks (32, 64, 128 filters; 2x Conv-BN-ReLU each; MaxPool), "
                     "GlobalAveragePooling2D, Dense(128), Dropout, Dense(1, sigmoid)"),
    "head_dropout": best_config["dropout"],
    "optimizer": "Adam", "learning_rate": best_config["learning_rate"],
    "loss": "binary_crossentropy", "batch_size": BATCH_SIZE,
    "epochs_requested": EPOCHS, "epochs_completed": len(history.epoch),
    "callbacks": {"early_stopping_patience": ES_PATIENCE, "reduce_lr_patience": RLR_PATIENCE},
    "class_weights": {str(k): v for k, v in class_weights.items()},
    "seed": SEED,
    "tuning": {"ran": RUN_TUNING, "subset_train": TUNE_TRAIN_SIZE, "subset_val": TUNE_VAL_SIZE,
               "epochs": TUNE_EPOCHS, "results": tuning_df.to_dict(orient="records"),
               "selected": best_config},
    "validation_results": val_metrics,
    "decision_threshold": val_metrics["tuned_threshold"],
    "decision_threshold_selection": "max macro-F1 on validation set",
    "training_samples": len(train_df), "validation_samples": len(val_df),
    "model_file": BEST_MODEL_PATH.name,
    "model_size_mb": round(BEST_MODEL_PATH.stat().st_size / (1024 ** 2), 2),
    "total_parameters": int(best_model.count_params()),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=4)
if len(tuning_df):
    tuning_df.to_csv(TUNING_PATH, index=False)

for p in [BEST_MODEL_PATH, HISTORY_PATH, CONFIG_PATH, TUNING_PATH, *eval_paths.values()]:
    if Path(p).exists():
        print(Path(p).name, f"{Path(p).stat().st_size / (1024 ** 2):.2f} MB")

## Push artifacts to GitHub
Requires a `GITHUB_TOKEN` secret (Kaggle: Add-ons > Secrets; Colab: the key icon). Model files are allowed by `.gitignore` inside `artifacts/models/`.

In [ ]:
def get_github_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except ImportError:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")


def push_to_github(required, optional, message):
    token = get_github_token()
    if not token:
        raise ValueError("GITHUB_TOKEN is missing!")

    for p in required:
        if not Path(p).exists():
            raise FileNotFoundError(p)
    files = [Path(p) for p in required] + [Path(p) for p in optional if Path(p).exists()]

    for fp in files:
        if fp.stat().st_size >= 100 * 1024 * 1024:
            raise ValueError(f"{fp.name} exceeds GitHub's 100 MB limit. Use Git LFS.")
        rel = str(fp.relative_to(REPO_DIR))
        if subprocess.run(["git", "check-ignore", "-q", rel], cwd=REPO_DIR).returncode == 0:
            raise RuntimeError(f"{rel} is ignored by .gitignore")

    subprocess.run(["git", "config", "--local", "user.name", GIT_USER_NAME], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "config", "--local", "user.email", GIT_USER_EMAIL], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "add", "--"] + [str(f.relative_to(REPO_DIR)) for f in files],
                   cwd=REPO_DIR, check=True)

    if subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=REPO_DIR).returncode != 0:
        subprocess.run(["git", "commit", "-m", message], cwd=REPO_DIR, check=True)
    else:
        print("No new changes to commit.")

    subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--rebase", "origin", "main"], cwd=REPO_DIR, check=True)

    askpass = Path("/tmp/github-askpass.sh")
    askpass.write_text("""#!/bin/sh
case "$1" in
    *sername*) printf '%s\\n' "$GITHUB_USERNAME" ;;
    *assword*) printf '%s\\n' "$GITHUB_TOKEN" ;;
    *) exit 1 ;;
esac
""")
    os.chmod(askpass, 0o700)

    env = os.environ.copy()
    env.update({"GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0",
                "GITHUB_TOKEN": token, "GITHUB_USERNAME": GITHUB_USERNAME})
    result = subprocess.run(["git", "push", "origin", "main"], cwd=REPO_DIR,
                            env=env, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("GitHub push failed. Check repository synchronization.")
    print(result.stdout or "Push OK.")
    print("Pushed:", [f.name for f in files])

In [ ]:
required = [BEST_MODEL_PATH, HISTORY_PATH, CONFIG_PATH]
optional = [TUNING_PATH, CURVES_PATH, *eval_paths.values(),
            EVAL_DIR / f"{PREFIX}_confusion_matrix.png",
            EVAL_DIR / f"{PREFIX}_roc_curve.png",
            EVAL_DIR / f"{PREFIX}_pipeline_samples.png"]
push_to_github(required, optional, f"Train and save Custom CNN model ({RUN_TAG})")